# Práctica 10 · Que no se escape lo que no debe salir

Un sistema de atención al cliente trabaja con dos clases de información peligrosa. La primera son
los datos personales de los clientes: nombres, documentos de identidad, teléfonos, correos,
cuentas bancarias. La segunda es la información interna de la empresa que no todo el mundo debe
ver: márgenes, costos, sueldos, procedimientos.

Las dos se filtran por caminos distintos y se atienden con herramientas distintas. Esta práctica
cubre las dos.

Vas a construir un detector de datos personales y medirlo, que es la parte que casi nadie hace.
Vas a ver que las herramientas más usadas de la industria dejan pasar documentos peruanos, y que
la forma de tachar los datos cambia mucho lo que el sistema puede seguir respondiendo. Después
vas a montar un control de acceso por permisos y comprobar, con el mismo índice, que la misma
pregunta devuelve cosas distintas según quién la haga.

Una aclaración importante antes de empezar: **todos los datos personales de esta práctica son
inventados**. Los nombres no corresponden a personas reales y los números de documento, tarjeta y
cuenta no son válidos. Trabajar con datos reales de clientes para practicar sería exactamente el
tipo de descuido que esta práctica busca evitar.

Las celdas se ejecutan en orden, una por una, con Shift + Enter.

In [1]:
%pip install --quiet spacy presidio-analyzer presidio-anonymizer
%pip install --quiet https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl

print("Listo.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Listo.


## 2. Comprobar que Ollama responde

Ollama es el programa que ejecuta los modelos de lenguaje dentro de tu computadora. Tiene que
estar encendido para que este cuaderno funcione, así que lo primero es confirmarlo.

Si algo falla, la salida de la celda te dice qué hacer según tu sistema operativo.

In [2]:
import platform
import sys

import requests

OLLAMA_URL = "http://localhost:11434"

print(f"Sistema: {platform.system()} {platform.machine()}")
print(f"Python:  {sys.version.split()[0]}\n")

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    respuesta.raise_for_status()
    modelos = sorted(m["name"] for m in respuesta.json()["models"])
    print(f"Ollama responde. Tienes {len(modelos)} modelos descargados:\n")
    for m in modelos:
        print(f"  - {m}")
except Exception as e:
    print(f"Ollama no responde en {OLLAMA_URL}")
    print(f"Detalle: {type(e).__name__}\n")
    if platform.system() == "Darwin":
        print("En Mac: abre la aplicación Ollama desde la carpeta Aplicaciones.")
        print("Debe aparecer su ícono en la barra de menús, arriba a la derecha.")
    elif platform.system() == "Windows":
        print("En Windows: busca Ollama en el menú Inicio y ábrelo.")
        print("Debe aparecer su ícono junto al reloj, abajo a la derecha.")
    else:
        print("Ejecuta 'ollama serve' en una terminal.")

Sistema: Darwin arm64
Python:  3.12.13



Ollama responde. Tienes 14 modelos descargados:

  - embeddinggemma:300m
  - gemma3:1b
  - gemma3:4b
  - gemma4:12b-mlx
  - gemma4:26b
  - gemma4:e4b
  - granite4.1:3b
  - granite4.1:8b
  - mxbai-embed-large:latest
  - nemotron-mini:4b
  - nomic-embed-text:latest
  - qwen3-4b-cs-ft:latest
  - qwen3:4b
  - shieldgemma:2b


## 3. Elegir el modelo según tu equipo

El modelo va a correr en tu máquina, así que la memoria que tengas importa. Un modelo grande
en un equipo chico no se rompe: simplemente tarda muchísimo y el sistema se pone lento.

Abajo hay tres opciones. Deja activa una sola, la que corresponda a tu computadora, y comenta
las demás poniéndoles un signo de gato al inicio de la línea. Si no sabes cuánta memoria
tienes, quédate con la opción A, que funciona en cualquier equipo.

El modelo de embeddings no se elige por equipo: es ligero y va igual en todos. Sí conviene saber
de dónde salió esa elección, y la respuesta es que está medida con documentos en español; en la
práctica 3 vas a reproducir la medición y a ver a los tres candidatos compitiendo.

In [3]:
# ---- Opción A: equipos de 8 GB de memoria o menos (descarga 3.3 GB) ---------
MODELO_LLM = "gemma3:4b"

# ---- Opción B: equipos de 16 GB de memoria (descarga 10 GB) ----------------
# MODELO_LLM = "gemma4:12b"        # Windows y Linux
# MODELO_LLM = "gemma4:12b-mlx"    # Mac con chip Apple (M1 en adelante), va más rápido

# ---- Opción C: equipos de 32 GB de memoria o más (descarga 17 GB) ----------
# MODELO_LLM = "gemma4:26b"        # Windows y Linux
# MODELO_LLM = "gemma4:26b-mlx"    # Mac con chip Apple

# El modelo de embeddings es ligero y es el mismo para todos. La elección está
# medida, no copiada de un tutorial: lo comprobamos en la práctica 3. Se eligió
# éste porque es el único de los tres que encuentra un pasaje en inglés cuando la
# pregunta va en español, algo que hace falta en cuanto el corpus mezcla idiomas.
MODELO_EMBEDDINGS = "embeddinggemma:300m"

print(f"Modelo de lenguaje:   {MODELO_LLM}")
print(f"Modelo de embeddings: {MODELO_EMBEDDINGS}")
print("\nSi alguno no aparece en la lista de la celda anterior, descárgalo con:")
print(f"   ollama pull {MODELO_LLM}")
print(f"   ollama pull {MODELO_EMBEDDINGS}")

Modelo de lenguaje:   gemma3:4b
Modelo de embeddings: embeddinggemma:300m

Si alguno no aparece en la lista de la celda anterior, descárgalo con:
   ollama pull gemma3:4b
   ollama pull embeddinggemma:300m


## 4. Por dónde se escapa la información

Antes del código conviene tener el mapa. En un sistema como el que has construido hay tres
momentos donde la información puede salir por donde no debe, y cada uno se atiende distinto.

**Al ingresar los documentos.** Todo lo que metes al índice queda ahí, en texto plano, listo para
ser recuperado. Si un documento traía el DNI de un cliente, ese DNI está ahora en tu base y
cualquier pregunta suficientemente parecida lo puede sacar. Aquí es donde va la detección y
redacción de datos personales, que es la primera mitad de esta práctica.

**Al buscar.** El índice no sabe quién está preguntando. Le da igual si quien consulta es un
cliente, un agente o el gerente general: devuelve lo más parecido. Si en el mismo índice conviven
las políticas públicas y la tabla de márgenes, un cliente puede terminar leyendo los márgenes.
Aquí va el control de acceso, que es la segunda mitad.

**Al generar la respuesta.** El modelo recibe el contexto y redacta. Si el contexto traía algo
sensible, lo va a usar. Las defensas de este punto ya las viste en la práctica de guardrails.

Hay un cuarto camino que en este curso está resuelto por construcción, y vale la pena nombrarlo
porque en la mayoría de los proyectos no lo está. Cuando usas un modelo alojado por un proveedor
externo, cada consulta viaja por la red con su contexto completo, es decir, con los fragmentos de
tus documentos dentro. El proveedor puede registrar esas consultas o mantenerlas en caché
temporal. Todo lo que has construido corre en tu máquina, así que ese riesgo no existe aquí; para
sectores regulados, ese suele ser el argumento decisivo a favor de modelos locales.

## 5. Los tickets de trabajo

Vamos a trabajar con cinco tickets de atención al cliente, del tipo que se acumula en cualquier
mesa de ayuda. Es material realista: la gente escribe su nombre completo, su documento y su
teléfono sin que nadie se lo pida.

Insisto en que los datos son inventados. Fíjate en cuántos datos personales distintos hay en tan
poco texto.

In [4]:
# Cinco tickets inventados, con los datos personales que aparecen de verdad en una
# mesa de ayuda peruana: DNI, RUC, celular, correo, dirección y número de cuenta.
TICKETS = [
    ("T-001", "Buenos días, soy María Fernanda Quispe Huamán, con DNI 45871203. "
              "Mi pedido 48213 no ha llegado. Mi celular es +51 987 654 321 y mi "
              "correo mfquispe@gmail.com. Vivo en Av. Arequipa 2345, Lince, Lima."),
    ("T-002", "Solicito factura a nombre de Textiles del Sur SAC, RUC 20548712345, "
              "domicilio fiscal Jr. Lampa 1150, Cercado de Lima. Contacto: "
              "Carlos Alberto Ramírez, carlos.ramirez@textilesdelsur.pe"),
    ("T-003", "Pagué con mi Visa terminada en 4471, número completo "
              "4539 1488 0343 6467. El cargo salió doble. Soy Ana Lucía Torres, "
              "DNI 09876543, teléfono 01-4567890."),
    ("T-004", "Mi esposa Rosa Mendoza compró un horno el 12 de marzo. "
              "Necesitamos la garantía. Nuestro número es 999888777."),
    ("T-005", "Reclamo del cliente Jorge Luis Vargas Llosa (DNI 41235678) "
              "sobre el pedido 51902. Reembolsar S/ 349.90 a la cuenta "
              "BCP 194-2345678-0-99. Correo: jvargas@hotmail.com"),
]

# Lo que una persona marcaría a mano como dato personal en cada ticket.
# Es el conjunto de referencia contra el que vamos a medir los detectores.
ESPERADO = {
    "T-001": ["María Fernanda Quispe Huamán", "45871203", "+51 987 654 321",
              "mfquispe@gmail.com", "Av. Arequipa 2345, Lince, Lima"],
    "T-002": ["Textiles del Sur SAC", "20548712345", "Jr. Lampa 1150",
              "Carlos Alberto Ramírez", "carlos.ramirez@textilesdelsur.pe"],
    "T-003": ["4539 1488 0343 6467", "Ana Lucía Torres", "09876543", "01-4567890"],
    "T-004": ["Rosa Mendoza", "999888777"],
    "T-005": ["Jorge Luis Vargas Llosa", "41235678", "194-2345678-0-99",
              "jvargas@hotmail.com"],
}

# El total de datos marcados a mano. Contra ese número se van a medir los detectores:
# sin una lista de referencia hecha por una persona, no hay forma de saber si un
# detector funciona o solo lo parece.
total = sum(len(v) for v in ESPERADO.values())
print(f"{len(TICKETS)} tickets con {total} datos personales marcados a mano.\n")
for tid, texto in TICKETS:
    print(f"{tid}  ({len(ESPERADO[tid])} datos)")
    print(f"   {texto}\n")

5 tickets con 20 datos personales marcados a mano.

T-001  (5 datos)
   Buenos días, soy María Fernanda Quispe Huamán, con DNI 45871203. Mi pedido 48213 no ha llegado. Mi celular es +51 987 654 321 y mi correo mfquispe@gmail.com. Vivo en Av. Arequipa 2345, Lince, Lima.

T-002  (5 datos)
   Solicito factura a nombre de Textiles del Sur SAC, RUC 20548712345, domicilio fiscal Jr. Lampa 1150, Cercado de Lima. Contacto: Carlos Alberto Ramírez, carlos.ramirez@textilesdelsur.pe

T-003  (4 datos)
   Pagué con mi Visa terminada en 4471, número completo 4539 1488 0343 6467. El cargo salió doble. Soy Ana Lucía Torres, DNI 09876543, teléfono 01-4567890.

T-004  (2 datos)
   Mi esposa Rosa Mendoza compró un horno el 12 de marzo. Necesitamos la garantía. Nuestro número es 999888777.

T-005  (4 datos)
   Reclamo del cliente Jorge Luis Vargas Llosa (DNI 41235678) sobre el pedido 51902. Reembolsar S/ 349.90 a la cuenta BCP 194-2345678-0-99. Correo: jvargas@hotmail.com



Ese conjunto marcado a mano es la pieza que hace posible todo lo demás. Sin él puedes decir que
tu detector "funciona bien", pero no puedes decir cuánto se le escapa, que es la única cifra que
importa cuando alguien pregunta si el sistema cumple con la ley.

Es el mismo principio de la práctica anterior aplicado a otro problema: si no tienes contra qué
comparar, no estás midiendo.

## 6. Primer intento: reconocimiento de entidades

La herramienta clásica para encontrar nombres de personas y lugares en un texto se llama
reconocimiento de entidades nombradas. Es un modelo entrenado para etiquetar qué partes de una
frase son personas, organizaciones o lugares.

Usamos spaCy con su modelo de español. Mira con atención lo que etiqueta y, sobre todo, lo que
etiqueta mal.

In [5]:
import spacy

# spaCy reconoce entidades: nombres de persona, lugares y organizaciones. Es lo
# primero que se prueba, y se verá enseguida qué tipo de dato NO alcanza a ver.
nlp = spacy.load("es_core_news_sm")

for tid, texto in TICKETS:
    entidades = [(e.text, e.label_) for e in nlp(texto).ents]
    print(f"{tid}: {len(entidades)} entidades")
    for t, etiqueta in entidades:
        print(f"    {etiqueta:<6} {t}")
    print()

T-001: 8 entidades
    LOC    Buenos días
    MISC   María Fernanda Quispe Huamán
    MISC   DNI 45871203
    MISC   Mi
    MISC   Mi celular
    MISC   Vivo en Av. Arequipa 2345
    LOC    Lince
    LOC    Lima

T-002: 7 entidades
    ORG    Solicito
    LOC    Textiles del Sur SAC
    MISC   Jr.
    MISC   Lampa 1150
    LOC    Cercado de Lima
    PER    Contacto
    PER    Carlos Alberto Ramírez

T-003: 3 entidades
    PER    Visa
    MISC   El cargo salió doble
    PER    Soy Ana Lucía Torres

T-004: 3 entidades
    PER    Rosa Mendoza
    PER    Necesitamos
    MISC   Nuestro número

T-005: 5 entidades
    PER    Jorge Luis Vargas Llosa
    MISC   DNI 41235678
    PER    Reembolsar
    ORG    BCP
    MISC   Correo: jvargas@hotmail.com



Hay dos problemas a la vista, y los dos importan.

El primero es que **no ve ningún número**. El DNI, el RUC, el teléfono, la tarjeta y la cuenta
bancaria le pasan por delante sin que los marque. Es esperable: el modelo fue entrenado para
reconocer nombres propios, no formatos de documento.

El segundo es más incómodo: **etiqueta como entidad cosas que no lo son**. "Buenos días" aparece
como lugar, "Solicito" como organización, "Visa" y "Necesitamos" como persona. Si redactaras a
ciegas con esta salida, tacharías el saludo del cliente y dejarías su DNI intacto.

Y hay un tercero, más sutil, que se ve mirando el ticket T-001 con cuidado: el nombre de la
clienta quedó etiquetado como `MISC`, no como `PER`. Volveremos a eso, porque tiene consecuencias.

## 7. Segundo intento: una herramienta especializada

Microsoft publica una biblioteca dedicada exactamente a este problema, llamada Presidio. Es la
que aparece en la mayoría de las guías del tema, y trae reconocedores para correos, teléfonos,
tarjetas y documentos de identidad de varios países.

Vale la pena probarla, sobre todo por lo que va a mostrar.

In [6]:
# Presidio es la herramienta de Microsoft hecha específicamente para datos personales.
# Trae detectores de correo, teléfono, tarjeta y documentos de varios países.
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# Hay que decirle expresamente que trabaje en español y con qué modelo; por omisión
# usa inglés, y con textos en español encuentra bastante menos.
proveedor = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "es", "model_name": "es_core_news_sm"}],
})
analizador = AnalyzerEngine(nlp_engine=proveedor.create_engine(),
                            supported_languages=["es"])

for tid, texto in TICKETS:
    hallazgos = analizador.analyze(text=texto, language="es")
    print(f"{tid}: {len(hallazgos)} hallazgos")
    for h in hallazgos:
        print(f"    {h.entity_type:<16} confianza {h.score:<5} {texto[h.start:h.end]}")
    print()

T-001: 6 hallazgos
    EMAIL_ADDRESS    confianza 1.0   mfquispe@gmail.com
    LOCATION         confianza 0.85  Buenos días
    LOCATION         confianza 0.85  Lince
    LOCATION         confianza 0.85  Lima
    URL              confianza 0.5   gmail.com
    PHONE_NUMBER     confianza 0.4   +51 987 654 321

T-002: 7 hallazgos
    EMAIL_ADDRESS    confianza 1.0   carlos.ramirez@textilesdelsur.pe
    ORGANIZATION     confianza 0.85  Solicito
    LOCATION         confianza 0.85  Textiles del Sur SAC
    LOCATION         confianza 0.85  Cercado de Lima
    PERSON           confianza 0.85  Contacto
    PERSON           confianza 0.85  Carlos Alberto Ramírez
    URL              confianza 0.5   textilesdelsur.pe

T-003: 4 hallazgos
    CREDIT_CARD      confianza 1.0   4539 1488 0343 6467
    PERSON           confianza 0.85  Visa
    PERSON           confianza 0.85  Soy Ana Lucía Torres
    PHONE_NUMBER     confianza 0.4   09876543

T-004: 2 hallazgos
    PERSON           confianza 0.85  Ros

Encuentra los correos con total confianza y detecta algunos teléfonos, pero fíjate en lo que no
aparece por ningún lado: **el DNI y el RUC**.

No es un defecto de la herramienta, es una consecuencia de cómo está construida: trae
reconocedores para los documentos de los países donde más se usa, y el documento nacional de
identidad peruano no está entre ellos. Un DNI son ocho dígitos seguidos, un patrón que no llama
la atención de nadie que no sepa qué está buscando.

La lección es directa y aplica a cualquier herramienta que compres para esto: **lo que detecta
depende de para qué mercado fue hecha**. Si tu operación es peruana, mexicana o colombiana,
tienes que verificar caso por caso qué documentos reconoce y agregar los que falten. Y para
verificarlo necesitas exactamente lo que armamos en la sección 5.

## 8. Tercer intento: patrones propios

Los datos que se le escapan a las dos herramientas anteriores tienen algo en común: son formatos
fijos. Un DNI son ocho dígitos. Un RUC son once que empiezan con 10 o con 20. Un celular peruano
empieza con 9 y tiene nueve dígitos.

Para formatos fijos no hace falta un modelo. Basta una expresión regular, que es una forma de
describir un patrón de texto. Son rápidas, no se equivocan y las escribes tú, que es justo lo que
hace falta cuando la herramienta comprada no conoce tu país.

In [7]:
import re

# Tercer detector, este escrito a mano. Una expresión regular es un patrón de texto:
# por ejemplo, un DNI peruano son exactamente ocho dígitos seguidos. Ninguna
# herramienta general conoce estos formatos, así que hay que describirlos.
PATRONES = {
    # El orden importa: los patrones más específicos van primero, para que un RUC
    # no se confunda con otra cosa y una tarjeta no se lea como varios números.
    "RUC":      r"\b(?:10|20)\d{9}\b",
    "TARJETA":  r"\b(?:\d{4}[\s-]?){3}\d{4}\b",
    "CUENTA":   r"\b\d{3}-\d{7}-\d-\d{2}\b",
    "DNI":      r"\b\d{8}\b",
    "CELULAR":  r"(?:\+51\s?)?9\d{2}[\s-]?\d{3}[\s-]?\d{3}\b",
    "FIJO":     r"\b0?1[\s-]?\d{7}\b",
    "CORREO":   r"\b[\w.+-]+@[\w-]+\.[\w.]+\b",
}

for tid, texto in TICKETS:
    hallazgos = [(m.group().strip(), tipo)
                 for tipo, patron in PATRONES.items()
                 for m in re.finditer(patron, texto)]
    print(f"{tid}: {len(hallazgos)} hallazgos")
    for t, tipo in hallazgos:
        print(f"    {tipo:<10} {t}")
    print()

T-001: 3 hallazgos
    DNI        45871203
    CELULAR    +51 987 654 321
    CORREO     mfquispe@gmail.com

T-002: 2 hallazgos
    RUC        20548712345
    CORREO     carlos.ramirez@textilesdelsur.pe

T-003: 3 hallazgos
    TARJETA    4539 1488 0343 6467
    DNI        09876543
    FIJO       01-4567890

T-004: 1 hallazgos
    CELULAR    999888777

T-005: 3 hallazgos
    CUENTA     194-2345678-0-99
    DNI        41235678
    CORREO     jvargas@hotmail.com



Ahora sí aparecen los documentos, los teléfonos, la tarjeta y la cuenta. Lo que no aparece, por
supuesto, es ningún nombre: una expresión regular no tiene forma de saber que "Rosa Mendoza" es
una persona y "Buenos días" no.

Es decir que las dos aproximaciones fallan en cosas exactamente opuestas. Midámoslo.

## 9. Cuánto encuentra cada uno

Vamos a comparar los tres métodos contra el conjunto marcado a mano. La pregunta es simple:
de los veinte datos personales que hay, ¿cuántos encuentra cada uno?

In [8]:
def normalizar(s):
    return re.sub(r"[\s-]", "", s).lower()


# Compara lo que encontró un detector contra la lista hecha a mano. La comparación es
# flexible a propósito: cuenta como acierto si uno contiene al otro, porque un
# detector puede marcar el nombre completo y otro solo el apellido.
def cobertura(hallazgos_por_ticket):
    """Cuántos de los datos marcados a mano quedaron cubiertos, y cuáles no."""
    encontrados, perdidos = 0, []
    for tid, esperados in ESPERADO.items():
        hallados = [normalizar(h) for h in hallazgos_por_ticket.get(tid, []) if h]
        for dato in esperados:
            n = normalizar(dato)
            if any(n in h or h in n for h in hallados):
                encontrados += 1
            else:
                perdidos.append((tid, dato))
    return encontrados, perdidos


# Los tres detectores corriendo sobre los mismos tickets, para poder compararlos en
# igualdad de condiciones.
por_spacy = {tid: [e.text for e in nlp(t).ents] for tid, t in TICKETS}
por_presidio = {tid: [t[h.start:h.end]
                      for h in analizador.analyze(text=t, language="es")]
                for tid, t in TICKETS}
por_regex = {tid: [m.group().strip()
                   for patron in PATRONES.values()
                   for m in re.finditer(patron, t)]
             for tid, t in TICKETS}
# Y un cuarto, que es simplemente juntar dos de ellos. Cada uno ve lo que al otro se
# le escapa: los patrones atrapan los números y spaCy atrapa los nombres.
combinado = {tid: por_regex[tid] + por_spacy[tid] for tid, _ in TICKETS}

METODOS = [
    ("spaCy (entidades)", por_spacy),
    ("Presidio", por_presidio),
    ("expresiones regulares", por_regex),
    ("regex + spaCy juntos", combinado),
]

print(f"{'método':>24} {'encuentra':>11} {'de':>4} {'cobertura':>11}")
print("-" * 54)
resultados = {}
for nombre, hallazgos in METODOS:
    n, perdidos = cobertura(hallazgos)
    resultados[nombre] = perdidos
    print(f"{nombre:>24} {n:>11} {total:>4} {n/total*100:>10.0f}%")

for nombre, _ in METODOS:
    print(f"\nLo que se le escapa a {nombre}:")
    if not resultados[nombre]:
        print("    nada")
    for tid, dato in resultados[nombre]:
        print(f"    {tid}  {dato}")

                  método   encuentra   de   cobertura
------------------------------------------------------
       spaCy (entidades)          11   20         55%
                Presidio          12   20         60%
   expresiones regulares          12   20         60%
    regex + spaCy juntos          20   20        100%

Lo que se le escapa a spaCy (entidades):
    T-001  +51 987 654 321
    T-001  mfquispe@gmail.com
    T-002  20548712345
    T-002  carlos.ramirez@textilesdelsur.pe
    T-003  4539 1488 0343 6467
    T-003  09876543
    T-003  01-4567890
    T-004  999888777
    T-005  194-2345678-0-99

Lo que se le escapa a Presidio:
    T-001  María Fernanda Quispe Huamán
    T-001  45871203
    T-002  20548712345
    T-002  Jr. Lampa 1150
    T-003  01-4567890
    T-004  999888777
    T-005  41235678
    T-005  194-2345678-0-99

Lo que se le escapa a expresiones regulares:
    T-001  María Fernanda Quispe Huamán
    T-001  Av. Arequipa 2345, Lince, Lima
    T-002  Textiles del Su

Ninguno de los tres métodos llega solo ni a dos tercios, y juntos cubren todo. Y no es
casualidad: si miras las listas de lo que se le escapa a cada uno, verás que **son
complementarias**. A las expresiones regulares se les escapan exclusivamente nombres y
direcciones; a spaCy y a Presidio se les escapan exclusivamente números.

Eso lleva a una recomendación concreta que puedes aplicar mañana: no elijas entre modelo o
patrones, usa los dos. El modelo pone lo que no se puede describir con una regla, y las reglas
ponen lo que el modelo no fue entrenado para ver.

Pero antes de celebrar ese 100%, hay que decir de dónde salió.

## 10. Ese 100% no vale lo que parece

Vuelve a mirar cómo se construyeron las expresiones regulares de la sección 8. Las escribí
mirando los cinco tickets. El patrón del DNI son ocho dígitos seguidos porque así aparece el DNI
en estos tickets; el del celular contempla los separadores que estos tickets usan.

Después medí la cobertura sobre esos mismos cinco tickets.

Eso es un razonamiento circular, y el 100% que produce no dice lo que parece decir. No dice "este
detector encuentra los datos personales". Dice "este detector encuentra los datos personales que
tenía delante cuando lo escribí", que es una afirmación mucho más pobre y bastante inútil.

El nombre técnico de esto es sobreajuste, y es la trampa más común al evaluar cualquier sistema:
usar el mismo material para construir y para medir. Da números excelentes y ninguna información.

La forma de saber cuánto vale de verdad un detector es probarlo con material que no vio. Vamos a
hacerlo, sin tocar una sola línea de los patrones.

In [9]:
# Tickets nuevos, con las variaciones que aparecen en la práctica real: el DNI
# escrito con guion, el carné de extranjería, un documento antiguo de 7 dígitos,
# teléfonos con paréntesis, cuentas interbancarias, correos mal tecleados y
# apellidos quechua, japoneses y chinos, todos frecuentes en el Perú.
# Siguen siendo datos inventados.
# Aquí viene la parte incómoda del cuaderno. Estos tickets NO se usaron para escribir
# los patrones, y traen las variaciones que aparecen en la realidad. Medir con el
# mismo material con el que se construyó el detector siempre da una cifra optimista.
NUEVOS = [
    ("N-01", "buenas, mi nombre es luis alberto ccahuana mamani, dni 4587120-3, "
             "el pedido no llega. mi cel es 9 87 654 321",
     ["luis alberto ccahuana mamani", "4587120-3", "9 87 654 321"]),
    ("N-02", "Soy Kenji Nakamura Flores, carné de extranjería 001234567, "
             "vivo en Calle Los Nogales 442, San Isidro. Tel (01) 456-7890",
     ["Kenji Nakamura Flores", "001234567", "Calle Los Nogales 442, San Isidro",
      "(01) 456-7890"]),
    ("N-03", "Necesito la boleta. Mi esposo Wong Chan se llama, y su documento "
             "es el 7654321 (tiene 7 dígitos, es antiguo). Escribir a "
             "wchan @ empresa.com.pe",
     ["Wong Chan", "7654321", "wchan @ empresa.com.pe"]),
    ("N-04", "Reclamo de la Sra. Yolanda Huarcaya Ttito, pasaporte N12345678, "
             "celular 51987654321, dirección Prolongación Bolognesi Mz. F Lt. 12, "
             "Villa El Salvador",
     ["Yolanda Huarcaya Ttito", "N12345678", "51987654321",
      "Prolongación Bolognesi Mz. F Lt. 12, Villa El Salvador"]),
    ("N-05", "Transferir el reembolso a la cuenta interbancaria "
             "002-193-001234567890-45 del titular Percy Quispe. "
             "Su whatsapp: +51-976-543-210",
     ["002-193-001234567890-45", "Percy Quispe", "+51-976-543-210"]),
    ("N-06", "Buenos días. Adjunto mi RUC 10456789012 y el correo de mi contadora "
             "maria.perez+facturas@estudio-contable.pe para el envío de facturas.",
     ["10456789012", "maria.perez+facturas@estudio-contable.pe"]),
]


def normalizar_amplio(s):
    return re.sub(r"[\s\-()]", "", s).lower()


# La misma medición de antes, pero aplicable a cualquier conjunto de prueba. Así se
# puede correr sobre los tickets conocidos y sobre los nuevos sin cambiar nada.
def medir(conjunto):
    """Cobertura del detector combinado sobre una lista de (id, texto, esperado)."""
    total = hits = 0
    perdidos = []
    for tid, texto, esperado in conjunto:
        hallados = [normalizar_amplio(h) for h in
                    [m.group().strip() for p in PATRONES.values()
                     for m in re.finditer(p, texto)]
                    + [e.text for e in nlp(texto).ents] if h]
        for dato in esperado:
            total += 1
            n = normalizar_amplio(dato)
            if any(n in h or h in n for h in hallados):
                hits += 1
            else:
                perdidos.append((tid, dato))
    return hits, total, perdidos


# El mismo detector, dos conjuntos. La diferencia entre las dos cifras es la que hay
# que reportar; la primera sola engaña, y es la que se suele presentar en una demo.
vistos = [(tid, texto, ESPERADO[tid]) for tid, texto in TICKETS]
h1, t1, _ = medir(vistos)
h2, t2, se_escapan = medir(NUEVOS)

print("EL MISMO DETECTOR, DOS CONJUNTOS DE PRUEBA\n")
print(f"  con los tickets que vio al escribirse : {h1}/{t1} = {h1/t1*100:.0f}%")
print(f"  con tickets que nunca había visto     : {h2}/{t2} = {h2/t2*100:.0f}%")
print(f"  caída: {(h1/t1 - h2/t2)*100:.0f} puntos\n")

print("Lo que se le escapa en el material nuevo:\n")
for tid, dato in se_escapan:
    print(f"   {tid}  {dato!r}")

EL MISMO DETECTOR, DOS CONJUNTOS DE PRUEBA

  con los tickets que vio al escribirse : 20/20 = 100%
  con tickets que nunca había visto     : 12/19 = 63%
  caída: 37 puntos

Lo que se le escapa en el material nuevo:

   N-01  '4587120-3'
   N-01  '9 87 654 321'
   N-02  '001234567'
   N-02  '(01) 456-7890'
   N-03  '7654321'
   N-03  'wchan @ empresa.com.pe'
   N-05  '002-193-001234567890-45'


Treinta y siete puntos de diferencia entre los dos números, y el segundo es el que se parece a la
realidad.

Mira lo que se escapa, porque no es aleatorio. Es el mismo dato escrito de otra forma: el DNI con
un guion en medio, el documento antiguo de siete dígitos, el carné de extranjería que tiene nueve,
el teléfono con paréntesis, la cuenta interbancaria con otro formato. Nada de eso es raro; es lo
que hay en cualquier bandeja de tickets reales.

Y hay un detalle que conviene mirar con cuidado, porque matiza la conclusión: **los nombres sí se
detectaron todos**, incluidos los escritos en minúsculas y los apellidos poco frecuentes. Lo que
falló fue la parte que escribimos nosotros a mano.

Tiene sentido si se piensa. El modelo de entidades fue entrenado con millones de textos que
nadie de nosotros eligió, así que generaliza a nombres que no vio. Las expresiones regulares las
escribimos mirando cinco ejemplos, así que solo cubren esos cinco ejemplos. El sobreajuste no
está repartido por igual: está concentrado en la parte hecha a medida.

De aquí salen dos hábitos que valen para cualquier sistema que midas, no solo para este:

**Separa el material con el que construyes del material con el que mides.** Si escribes las
reglas mirando unos datos, mide con otros. Es incómodo porque los números bajan, pero los
números que bajan son los verdaderos.

**Desconfía de las cifras redondas.** Un 100% en una evaluación casi nunca significa que el
sistema sea perfecto; casi siempre significa que la prueba era el material de desarrollo. Cuando
veas ese número en un informe, la primera pregunta es con qué datos se obtuvo y de dónde
salieron.

## 11. Y qué se hace con eso

La reacción natural ante el 63% es agregar patrones: uno para el DNI con guion, otro para el
carné de extranjería, otro para la cuenta interbancaria. Es lo correcto, y hay que hacerlo.

Pero conviene hacerlo sabiendo lo que pasa después: en cuanto agregues esos patrones mirando
estos seis tickets nuevos, el 63% va a subir a casi 100%, y ese número tampoco va a valer nada,
por exactamente la misma razón de antes. Habrás sobreajustado al segundo conjunto.

La salida no es dejar de ajustar, es cambiar cómo se mide. En la práctica se hace así: se aparta
un conjunto de textos que no se toca nunca, se ajusta el detector con los demás, y se mide con el
apartado solo cuando ya no se va a cambiar nada más. Si además el corpus va creciendo, se aparta
material nuevo cada cierto tiempo, porque la forma en que la gente escribe cambia.

Para un sistema de atención al cliente en operación hay una versión práctica y barata de esto:
tomar cien tickets al azar cada trimestre, marcarlos a mano y medir. Es una tarde de trabajo y es
la única cifra que se puede llevar a una auditoría sin sonrojarse.

## 12. Tres formas de tachar, y no dan lo mismo

Detectar es la mitad del trabajo. Falta decidir qué se hace con lo detectado, y ahí hay tres
caminos que se usan en la práctica.

**Enmascarar**: reemplazar el dato por un relleno genérico, como `XXXX`.

**Nulificar**: borrarlo, dejando el hueco.

**Sustituir por su tipo**: reemplazarlo por una etiqueta que dice qué clase de dato era, como
`[DNI]` o `[NOMBRE]`.

Las tres protegen igual de bien, porque en las tres el dato desaparece. La diferencia está en lo
que queda después, y eso decide si tu sistema puede seguir trabajando con el texto.

In [10]:
# Detectar no basta: para tachar hace falta saber DÓNDE está cada dato. Esta función
# devuelve la posición de inicio y fin de cada hallazgo, además de su tipo.
def hallar_todo(texto):
    """Todos los tramos con dato personal: (inicio, fin, tipo)."""
    tramos = []
    for tipo, patron in PATRONES.items():
        for m in re.finditer(patron, texto):
            tramos.append((m.start(), m.end(), tipo))
    for e in nlp(texto).ents:
        if e.label_ in ("PER", "LOC", "ORG"):
            tipo = {"PER": "NOMBRE", "LOC": "DIRECCION", "ORG": "EMPRESA"}[e.label_]
            tramos.append((e.start_char, e.end_char, tipo))

    # Un mismo trozo puede haber sido marcado dos veces. Nos quedamos con el
    # tramo más largo y descartamos lo que se le solape.
    tramos.sort(key=lambda t: (t[0], -(t[1] - t[0])))
    limpios, fin_previo = [], -1
    for inicio, fin, tipo in tramos:
        if inicio >= fin_previo:
            limpios.append((inicio, fin, tipo))
            fin_previo = fin
    return limpios


# Reconstruye el texto pegando los tramos limpios y sustituyendo los sensibles. Los
# tres modos borran la misma información pero dejan pistas distintas, y eso cambia
# lo que el modelo puede responder después.
def redactar(texto, modo):
    partes, cursor = [], 0
    for inicio, fin, tipo in hallar_todo(texto):
        partes.append(texto[cursor:inicio])
        partes.append({"enmascarar": "XXXX",
                       "nulificar": "",
                       "por_tipo": f"[{tipo}]"}[modo])
        cursor = fin
    partes.append(texto[cursor:])
    return "".join(partes)


tid, texto = TICKETS[1]
print(f"{tid} original:\n   {texto}\n")
for modo, nombre in (("enmascarar", "enmascarado"),
                     ("nulificar", "nulificado"),
                     ("por_tipo", "sustituido por tipo")):
    print(f"{tid} {nombre}:\n   {' '.join(redactar(texto, modo).split())}\n")

T-002 original:
   Solicito factura a nombre de Textiles del Sur SAC, RUC 20548712345, domicilio fiscal Jr. Lampa 1150, Cercado de Lima. Contacto: Carlos Alberto Ramírez, carlos.ramirez@textilesdelsur.pe

T-002 enmascarado:
   XXXX factura a nombre de XXXX, RUC XXXX, domicilio fiscal Jr. Lampa 1150, XXXX. XXXX: XXXX, XXXX

T-002 nulificado:
   factura a nombre de , RUC , domicilio fiscal Jr. Lampa 1150, . : ,

T-002 sustituido por tipo:
   [EMPRESA] factura a nombre de [DIRECCION], RUC [RUC], domicilio fiscal Jr. Lampa 1150, [DIRECCION]. [NOMBRE]: [NOMBRE], [CORREO]



Léelas seguidas y la diferencia salta.

La versión nulificada queda rota: "factura a nombre de , RUC , domicilio fiscal ... . : ,". No se
entiende ni qué se estaba pidiendo. Perdió el dato y perdió también la frase.

La enmascarada conserva la estructura pero no dice nada: sabes que había algo, no qué clase de
algo. "a nombre de XXXX" podría ser una persona o una empresa.

La sustituida por tipo conserva las dos cosas: "a nombre de [EMPRESA], RUC [RUC]". El dato se
fue, pero el sentido de la frase sobrevivió. Ese es el punto de esta estrategia: proteger sin
destruir la relación entre las cosas.

Comprobemos que no es solo una impresión al leer.

In [11]:
# La prueba que importa: después de tachar, ¿el sistema todavía sirve? Proteger datos
# es fácil si se acepta un sistema inútil; lo difícil es conservar la utilidad.
PLANTILLA = """Responde la pregunta usando únicamente estos tickets de atención.
Si el dato fue removido del texto, dilo en vez de inventarlo.

Tickets:
{contexto}

Pregunta: {pregunta}
Respuesta breve:"""

PREGUNTAS = [
    "¿Qué tipo de dato de contacto dejó el cliente del ticket T-001?",
    "¿La factura del ticket T-002 se pide a nombre de una persona o de una empresa?",
    "¿A qué medio se debe hacer el reembolso del ticket T-005?",
]

import ollama
# think=False más abajo es necesario con los modelos que razonan antes de contestar:
# sin eso gastan todo su presupuesto pensando y devuelven una respuesta vacía.
cliente = ollama.Client()

for modo in ("nulificar", "enmascarar", "por_tipo"):
    contexto = "\n\n".join(f"{tid}: {redactar(t, modo)}" for tid, t in TICKETS)
    print("=" * 70)
    print(f"estrategia: {modo}")
    print("=" * 70)
    for pregunta in PREGUNTAS:
        salida = cliente.generate(
            model=MODELO_LLM,
            prompt=PLANTILLA.format(contexto=contexto, pregunta=pregunta),
            think=False,
            options={"temperature": 0, "num_predict": 120},
        )["response"]
        print(f"\n  P: {pregunta}")
        print(f"  R: {' '.join(salida.split())[:200]}")
    print()

print("\nCuánto texto sobrevive a cada estrategia:\n")
# Cuánto texto sobrevive a cada estrategia. Nulificar borra más, pero también borra la
# pista de que ahí había algo, y el modelo ya no puede decir qué falta.
original = sum(len(t) for _, t in TICKETS)
print(f"{'estrategia':>14} {'caracteres':>12} {'del original':>14}")
print("-" * 44)
print(f"{'sin redactar':>14} {original:>12} {'100%':>14}")
for modo in ("nulificar", "enmascarar", "por_tipo"):
    n = sum(len(redactar(t, modo)) for _, t in TICKETS)
    print(f"{modo:>14} {n:>12} {n/original*100:>13.0f}%")

estrategia: nulificar



  P: ¿Qué tipo de dato de contacto dejó el cliente del ticket T-001?
  R: El cliente del ticket T-001 dejó su nombre (María Fernanda Quispe Huamán), DNI, celular y correo electrónico.



  P: ¿La factura del ticket T-002 se pide a nombre de una persona o de una empresa?
  R: La factura se pide a nombre de , RUC .



  P: ¿A qué medio se debe hacer el reembolso del ticket T-005?
  R: La respuesta se debe hacer a la cuenta .

estrategia: enmascarar



  P: ¿Qué tipo de dato de contacto dejó el cliente del ticket T-001?
  R: Celular (XXXX), correo electrónico (XXXX) y dirección (Av. Arequipa 2345, XXXX, XXXX).



  P: ¿La factura del ticket T-002 se pide a nombre de una persona o de una empresa?
  R: La factura del ticket T-002 se pide a nombre de una empresa (RUC XXXX).



  P: ¿A qué medio se debe hacer el reembolso del ticket T-005?
  R: La cuenta XXXX XXXX.

estrategia: por_tipo



  P: ¿Qué tipo de dato de contacto dejó el cliente del ticket T-001?
  R: María Fernanda Quispe Huamán proporcionó su celular ([CELULAR]) y correo ([CORREO]).



  P: ¿La factura del ticket T-002 se pide a nombre de una persona o de una empresa?
  R: La factura del ticket T-002 se pide a nombre de una empresa.



  P: ¿A qué medio se debe hacer el reembolso del ticket T-005?
  R: Con mi tarjeta [TARJETA] terminada en 4471.


Cuánto texto sobrevive a cada estrategia:

    estrategia   caracteres   del original
--------------------------------------------
  sin redactar          805           100%
     nulificar          456            57%
    enmascarar          564            70%
      por_tipo          678            84%


Con el texto nulificado, la pregunta por el medio de reembolso recibe una respuesta que no
resuelve nada. Con el texto enmascarado, algo se puede decir pero queda opaco. Con la sustitución
por tipo, el sistema responde que es una cuenta bancaria, que es exactamente lo que se necesitaba
saber sin necesidad de conocer el número.

La tabla del final pone cifras a lo que ya se veía: nulificar se lleva casi la mitad del texto y
sustituir por tipo conserva la mayor parte.

Para atención al cliente la elección es clara. Y hay un argumento adicional que no es de calidad
sino de auditoría: cuando alguien revise tu corpus redactado, con la sustitución por tipo puede
verificar que la redacción se aplicó y a qué; con `XXXX` en todos lados, no puede distinguir un
dato tachado de otro.

## 13. Lo que se escapó, y por qué

Vuelve a mirar el ticket T-001 redactado por cualquiera de las tres estrategias. Hay algo que no
debería seguir ahí.

In [12]:
tid, texto = TICKETS[0]
print(f"{tid} sustituido por tipo:")
print(f"   {' '.join(redactar(texto, 'por_tipo').split())}\n")
print("Etiquetas que spaCy pone en la primera frase:\n")
for e in nlp(texto[:75]).ents:
    print(f"   {e.label_:<6} {e.text!r}")

T-001 sustituido por tipo:
   [DIRECCION], soy María Fernanda Quispe Huamán, con DNI [DNI]. Mi pedido 48213 no ha llegado. Mi celular es [CELULAR] y mi correo [CORREO]. Vivo en Av. Arequipa 2345, [DIRECCION], [DIRECCION].

Etiquetas que spaCy pone en la primera frase:

   LOC    'Buenos días'
   MISC   'María Fernanda Quispe Huamán'
   MISC   'DNI 45871203'
   MISC   'Mi'


El nombre completo de la clienta sobrevivió a la redacción.

La causa está en la última salida: spaCy etiquetó "María Fernanda Quispe Huamán" como `MISC`, una
categoría de entidades misceláneas, y no como `PER`. Nuestro filtro solo acepta `PER`, `LOC` y
`ORG`, así que el nombre pasó de largo. Mientras tanto, "Buenos días" sí entró como `DIRECCION`,
porque el modelo lo etiquetó como lugar.

O sea que el sistema tachó el saludo y dejó el nombre. Es un fallo silencioso: nada se rompe,
ningún error aparece, y el texto redactado se ve perfectamente razonable hasta que lo lees con
cuidado.

La solución obvia es aceptar también `MISC`. Veamos qué cuesta.

In [13]:
# Un detector demasiado ansioso también es un problema: si tacha palabras que no son
# datos personales, destruye el texto. Aquí se mide ese otro lado del error.
NOMBRES = ["María Fernanda Quispe Huamán", "Carlos Alberto Ramírez",
           "Jorge Luis Vargas Llosa", "Rosa Mendoza", "Ana Lucía Torres"]

# Todo lo que sí queríamos tachar, para no contarlo como error del detector.
LEGITIMO = {normalizar(d) for datos in ESPERADO.values() for d in datos}


# Distingue lo que sí queríamos tachar de lo que se tachó de más. Sin esta distinción,
# un detector que tache todo saldría con 100% de cobertura.
def es_dato_real(entidad):
    n = normalizar(entidad)
    if any(n in d or d in n for d in LEGITIMO):
        return True
    return any(re.search(p, entidad) for p in PATRONES.values())


# La misma corrida aceptando una etiqueta más. Se gana cobertura y se pierde precisión:
# es la decisión que hay que tomar con números delante, no por intuición.
for etiquetas in (("PER", "LOC", "ORG"), ("PER", "LOC", "ORG", "MISC")):
    detectados, ruido = 0, []
    for tid, texto in TICKETS:
        ents = [e.text for e in nlp(texto).ents if e.label_ in etiquetas]
        juntas = " ".join(ents)
        for n in NOMBRES:
            if all(p in juntas for p in n.split()[:2]):
                detectados += 1
                break
        ruido += [e for e in ents if not es_dato_real(e)]
    print(f"aceptando {etiquetas}:")
    print(f"   nombres completos detectados: {detectados} de {len(NOMBRES)}")
    print(f"   palabras que se tacharían sin ser dato personal: {len(ruido)}")
    for e in ruido:
        print(f"      {e!r}")
    print()

aceptando ('PER', 'LOC', 'ORG'):
   nombres completos detectados: 4 de 5
   palabras que se tacharían sin ser dato personal: 8
      'Buenos días'
      'Solicito'
      'Cercado de Lima'
      'Contacto'
      'Visa'
      'Necesitamos'
      'Reembolsar'
      'BCP'

aceptando ('PER', 'LOC', 'ORG', 'MISC'):
   nombres completos detectados: 5 de 5
   palabras que se tacharían sin ser dato personal: 12
      'Buenos días'
      'Mi celular'
      'Vivo en Av. Arequipa 2345'
      'Solicito'
      'Cercado de Lima'
      'Contacto'
      'Visa'
      'El cargo salió doble'
      'Necesitamos'
      'Nuestro número'
      'Reembolsar'
      'BCP'



Aceptar `MISC` recupera el nombre que faltaba, y a cambio tacha algunas palabras más que no son
datos personales.

Esa es la decisión de fondo de cualquier sistema de este tipo, y no tiene una respuesta técnica.
Es una decisión de negocio: ¿qué prefieres, que se te escape un dato personal de vez en cuando, o
que se tachen palabras normales y el texto quede más pobre?

Para datos de clientes bajo una ley de protección de datos, el error caro es dejar pasar. Más
vale un corpus con algunas palabras tachadas de más que una multa. Para un archivo interno donde
la prioridad es que el texto siga siendo útil, la balanza puede inclinarse al otro lado.

Lo que no es admisible es tomar esa decisión sin saber que la estás tomando, que es lo que pasa
cuando uno usa la configuración por omisión de una biblioteca y no la mide.

## 14. La segunda mitad: quién puede ver qué

Cambiamos de problema. Los datos personales ya están tratados; ahora se trata de la información
interna de la empresa.

El punto de partida es una propiedad del índice que conviene tener muy presente: **el índice no
sabe quién pregunta**. Cuando le pides los fragmentos más parecidos, te los da. No tiene noción
de roles ni de permisos, igual que no tenía noción de "no sé" en la práctica anterior.

Montemos un corpus pequeño con tres niveles de confidencialidad para verlo.

In [14]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

from langchain_community.vectorstores import LanceDB
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

# Cambia el tema: ya no se trata de tachar datos, sino de que cada persona vea solo
# lo que le corresponde. Seis documentos en tres niveles de confidencialidad.
BASE = [
    ("POL-DEV", "publico", "Política de devoluciones: el cliente tiene 30 días "
     "naturales desde la entrega para solicitar la devolución de un producto."),
    ("POL-ENV", "publico", "Plazos de envío: el envío estándar tarda de 3 a 5 días "
     "hábiles y cuesta S/ 12.90, gratis desde S/ 149.00 en Lima."),
    ("PRO-ESC", "interno", "Procedimiento de escalamiento: si el cliente insiste "
     "tras dos respuestas, el agente transfiere al supervisor de turno."),
    ("PRO-DES", "interno", "Procedimiento de descuentos: el agente puede otorgar "
     "hasta 10% de descuento sin autorización; más requiere al jefe de tienda."),
    ("CON-MAR", "confidencial", "Margen por categoría: electrodomésticos 22%, "
     "textiles 47%, calzado 39%. El costo de adquisición del horno TS-HOR-88 "
     "es S/ 410.00 y se vende en S/ 749.00."),
    ("CON-NOM", "confidencial", "Nómina del área de atención: el sueldo base de "
     "un agente es S/ 2,300 y el del supervisor S/ 4,100."),
]

# El nivel viaja en los metadatos de cada fragmento. Ese dato es el que permitirá
# filtrar después; si no se guarda al indexar, ya no hay forma de recuperarlo.
documentos = [Document(page_content=texto,
                       metadata={"doc_id": doc_id, "nivel": nivel})
              for doc_id, nivel, texto in BASE]

embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)
almacen = LanceDB.from_documents(documentos, embeddings,
                                 uri="/tmp/lancedb_practica10",
                                 table_name="permisos", mode="overwrite")

print(f"{len(documentos)} documentos indexados, con su nivel en los metadatos:\n")
for doc_id, nivel, _ in BASE:
    print(f"   {doc_id:<10} {nivel}")

6 documentos indexados, con su nivel en los metadatos:

   POL-DEV    publico
   POL-ENV    publico
   PRO-ESC    interno
   PRO-DES    interno
   CON-MAR    confidencial
   CON-NOM    confidencial


Ahora las preguntas peligrosas, sin ningún control.

In [15]:
PREGUNTAS = [
    "¿Cuánto cuesta el envío estándar?",
    "¿Cuánto descuento puede dar un agente sin pedir permiso?",
    "¿Cuál es el margen de los electrodomésticos?",
    "¿Cuánto gana un supervisor?",
]

print("Lo que devuelve el índice sin filtro alguno:\n")
for pregunta in PREGUNTAS:
    d = almacen.similarity_search(pregunta, k=1)[0]
    print(f"  {pregunta}")
    print(f"     -> {d.metadata['doc_id']} ({d.metadata['nivel']})")
    print(f"        {d.page_content[:76]}...")
    print()

Lo que devuelve el índice sin filtro alguno:

  ¿Cuánto cuesta el envío estándar?
     -> POL-ENV (publico)
        Plazos de envío: el envío estándar tarda de 3 a 5 días hábiles y cuesta S/ 1...



  ¿Cuánto descuento puede dar un agente sin pedir permiso?
     -> PRO-DES (interno)
        Procedimiento de descuentos: el agente puede otorgar hasta 10% de descuento ...

  ¿Cuál es el margen de los electrodomésticos?
     -> CON-MAR (confidencial)
        Margen por categoría: electrodomésticos 22%, textiles 47%, calzado 39%. El c...

  ¿Cuánto gana un supervisor?
     -> CON-NOM (confidencial)
        Nómina del área de atención: el sueldo base de un agente es S/ 2,300 y el de...



Las dos últimas preguntas devuelven documentos confidenciales. Si esas preguntas las escribe un
cliente en el chat de la tienda, el sistema le va a redactar una respuesta con los márgenes de la
empresa y con los sueldos del personal.

No hace falta que el cliente sea malicioso ni que sepa lo que está haciendo. Basta con que
pregunte algo cuyas palabras se parezcan a las de un documento que no debía estar a su alcance.

## 15. Filtrar por permisos

El arreglo es filtrar la búsqueda por los metadatos, dejando fuera del alcance los niveles a los
que quien pregunta no tiene acceso.

Lo importante es *cuándo* se filtra. Hay dos momentos posibles y solo uno sirve:

Si filtras **después** de buscar, el índice primero encuentra los tres fragmentos más parecidos y
luego tú descartas los prohibidos. Funciona, pero si los tres eran confidenciales te quedas sin
nada, y además el documento prohibido ya pasó por tu código.

Si filtras **antes**, el índice busca únicamente entre los documentos permitidos. Siempre
devuelve lo mejor de lo que esa persona sí puede ver. Eso es lo que hace `prefilter=True`.

Un detalle de LanceDB que conviene anotar porque cuesta descubrirlo: los metadatos se guardan
agrupados, así que en la condición hay que escribir `metadata.nivel` y no solo `nivel`.

In [16]:
# Qué niveles puede ver cada rol. Es una tabla de negocio, no una decisión técnica:
# la escribe quien manda en la empresa, no quien programa.
PERMISOS = {
    "cliente": ["publico"],
    "agente": ["publico", "interno"],
    "gerente": ["publico", "interno", "confidencial"],
}


# La clave está en prefilter=True: el filtro se aplica ANTES de buscar. Si se aplicara
# después, el sistema habría leído los documentos prohibidos y solo los estaría
# escondiendo, que es una diferencia enorme ante una auditoría.
def buscar_como(rol, pregunta, k=1):
    condicion = " OR ".join(f"metadata.nivel = '{n}'" for n in PERMISOS[rol])
    return almacen.similarity_search(pregunta, k=k, filter=condicion, prefilter=True)


print("El mismo índice, la misma pregunta, distinto rol:\n")
print(f"{'pregunta':<50} {'cliente':>10} {'agente':>9} {'gerente':>10}")
print("-" * 81)
for pregunta in PREGUNTAS:
    fila = []
    for rol in PERMISOS:
        r = buscar_como(rol, pregunta)
        fila.append(r[0].metadata["doc_id"] if r else "—")
    print(f"{pregunta[:48]:<50} {fila[0]:>10} {fila[1]:>9} {fila[2]:>10}")

El mismo índice, la misma pregunta, distinto rol:

pregunta                                              cliente    agente    gerente
---------------------------------------------------------------------------------


¿Cuánto cuesta el envío estándar?                     POL-ENV   POL-ENV    POL-ENV


¿Cuánto descuento puede dar un agente sin pedir       POL-DEV   PRO-DES    PRO-DES


¿Cuál es el margen de los electrodomésticos?          POL-ENV   PRO-DES    CON-MAR


¿Cuánto gana un supervisor?                           POL-ENV   PRO-DES    CON-NOM


La columna del gerente es la de antes. La del cliente ya no contiene ni un documento
confidencial: pregunte lo que pregunte, solo se le busca entre los públicos.

Veamos con detalle qué recibe cada rol ante la pregunta por el margen.

In [17]:
pregunta = "¿Cuál es el margen de los electrodomésticos?"
print(f"{pregunta}\n")
for rol in PERMISOS:
    print(f"  {rol} ({', '.join(PERMISOS[rol])}):")
    for d in buscar_como(rol, pregunta, k=2):
        print(f"     {d.metadata['doc_id']} ({d.metadata['nivel']}): "
              f"{d.page_content[:66]}...")
    print()

¿Cuál es el margen de los electrodomésticos?

  cliente (publico):
     POL-ENV (publico): Plazos de envío: el envío estándar tarda de 3 a 5 días hábiles y c...
     POL-DEV (publico): Política de devoluciones: el cliente tiene 30 días naturales desde...

  agente (publico, interno):
     PRO-DES (interno): Procedimiento de descuentos: el agente puede otorgar hasta 10% de ...
     POL-ENV (publico): Plazos de envío: el envío estándar tarda de 3 a 5 días hábiles y c...

  gerente (publico, interno, confidencial):


     CON-MAR (confidencial): Margen por categoría: electrodomésticos 22%, textiles 47%, calzado...
     PRO-DES (interno): Procedimiento de descuentos: el agente puede otorgar hasta 10% de ...



Aquí aparece un problema que el filtro no resuelve, y que conviene ver ahora y no en producción.

El cliente preguntó por el margen de los electrodomésticos y recibió los plazos de envío. No es
una fuga: el filtro hizo su trabajo. Pero tampoco es una respuesta. Es el mismo comportamiento de
siempre, el índice devolviendo lo más parecido de lo que tiene disponible, que ahora es un
conjunto recortado.

Si el sistema redacta con ese contexto, el cliente va a recibir un texto sobre envíos ante una
pregunta sobre márgenes. Queda confundido y probablemente vuelva a preguntar.

Lo correcto es combinar el filtro con lo que aprendiste en la práctica anterior: si lo que se
recuperó no responde la pregunta, decirlo. Y aquí hay un matiz de negocio, porque no todas las
negativas son iguales. Ante una pregunta sobre márgenes, un sistema puede responder "no tengo esa
información" o puede responder "esa información no está disponible por este canal". La segunda es
más honesta, pero también confirma que la información existe. Cuál conviene depende de qué tan
sensible sea el dato, y esa decisión no es del área de sistemas.

## 16. Dejar constancia

Falta una pieza que no protege nada pero sin la cual no se puede demostrar nada: el registro.

Cuando alguien pregunte si el sistema entregó un dato que no debía, o cuando toque una auditoría,
la respuesta no puede ser una impresión. Tiene que haber un registro de qué se preguntó, quién
preguntó, qué se recuperó y qué se respondió.

Hay una tensión evidente y hay que resolverla a propósito: ese registro contiene las preguntas de
los clientes, y las preguntas de los clientes traen datos personales. Un registro de auditoría
mal hecho es una segunda copia sin proteger de todo lo que acabas de redactar. Por eso lo que se
guarda pasa por el mismo redactor.

Sobre el hash: sirve para poder afirmar más tarde que un registro no fue modificado. Si alguien
edita la línea, el hash deja de coincidir con su contenido. Es la base de lo que las normas de
seguridad de la información llaman trazabilidad, y cuesta tres líneas ponerlo.

In [18]:
import hashlib
import json
from datetime import datetime, timezone

# Tercer tema: dejar constancia. Sin registro no se puede responder a la pregunta que
# hace cualquier auditoría, que es quién consultó qué y cuándo.
BITACORA = []


# El identificador del usuario se guarda como huella, no como correo: alcanza para
# saber que fue la misma persona sin guardar quién es.
def consultar_con_registro(rol, pregunta, usuario):
    recuperados = buscar_como(rol, pregunta, k=2)
    contexto = "\n\n".join(d.page_content for d in recuperados)
    respuesta = cliente.generate(
        model=MODELO_LLM,
        prompt=f"Responde usando solo este contexto. Si no está, dilo.\n\n"
               f"{contexto}\n\nPregunta: {pregunta}\nRespuesta:",
        think=False,
        options={"temperature": 0, "num_predict": 120},
    )["response"].strip()

    entrada = {
        # La pregunta se guarda redactada: el registro no puede convertirse en
        # una segunda copia de los datos personales que acabamos de proteger.
        "momento": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "usuario": hashlib.sha256(usuario.encode()).hexdigest()[:12],
        "rol": rol,
        "pregunta": redactar(pregunta, "por_tipo"),
        "documentos": [d.metadata["doc_id"] for d in recuperados],
        "niveles": sorted({d.metadata["nivel"] for d in recuperados}),
        "respuesta": redactar(respuesta, "por_tipo")[:160],
    }
    # Una huella del registro completo. Si alguien edita una entrada después, el
    # recálculo ya no coincide y la alteración se nota.
    entrada["huella"] = hashlib.sha256(
        json.dumps(entrada, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()[:16]
    BITACORA.append(entrada)
    return respuesta


consultar_con_registro("cliente", "¿Cuánto cuesta el envío estándar?", "mfquispe@gmail.com")
consultar_con_registro("cliente", "¿Cuál es el margen de los electrodomésticos?", "mfquispe@gmail.com")
consultar_con_registro("gerente", "¿Cuál es el margen de los electrodomésticos?", "jefe@tiendasol.pe")

print(f"{len(BITACORA)} consultas registradas.\n")
for e in BITACORA:
    print(f"  {e['momento']}  usuario {e['usuario']}  rol {e['rol']}")
    print(f"     pregunta   : {e['pregunta']}")
    print(f"     recuperó   : {e['documentos']}  niveles {e['niveles']}")
    print(f"     respondió  : {' '.join(e['respuesta'].split())[:110]}")
    print(f"     huella     : {e['huella']}")
    print()

3 consultas registradas.

  2026-08-07T18:53:19+00:00  usuario 7229504ff79d  rol cliente
     pregunta   : [EMPRESA] cuesta el envío estándar?
     recuperó   : ['POL-ENV', 'POL-DEV']  niveles ['publico']
     respondió  : [NOMBRE] 12.90
     huella     : 4b9fd7c871412d01

  2026-08-07T18:53:20+00:00  usuario 7229504ff79d  rol cliente
     pregunta   : [DIRECCION] es el margen de los electrodomésticos?
     recuperó   : ['POL-ENV', 'POL-DEV']  niveles ['publico']
     respondió  : No está disponible información sobre el margen de los electrodomésticos en este contexto.
     huella     : c7c5958004df5a77

  2026-08-07T18:53:20+00:00  usuario 5db7946f2b95  rol gerente
     pregunta   : [DIRECCION] es el margen de los electrodomésticos?
     recuperó   : ['CON-MAR', 'PRO-DES']  niveles ['confidencial', 'interno']
     respondió  : 22%.
     huella     : e2eaece1570dba4c



Antes de seguir, lee las preguntas registradas. Hay un problema evidente.

"¿Cuánto cuesta el envío estándar?" quedó guardada como "[EMPRESA] cuesta el envío estándar", y
"¿Cuál es el margen..." como "[DIRECCION] es el margen...". El redactor tachó las palabras
"Cuánto" y "Cuál". Si la respuesta traía un monto en soles, seguramente verás que "S/" quedó
convertido en "[NOMBRE]".

No es un error de programación: es el mismo redactor que funcionaba bien sobre los tickets,
aplicado a un texto de otra naturaleza. Una pregunta de ocho palabras no le da al modelo de
entidades ningún contexto, y sin contexto etiqueta cualquier palabra que empiece con mayúscula.

La lección es que **un redactor se calibra para un tipo de texto y no se traslada solo a otro**.
Los tickets son párrafos con nombres reales dentro; las consultas son frases cortas sin casi
ningún dato personal. Para el registro conviene un redactor más conservador, por ejemplo solo con
las expresiones regulares, que sobre texto corto no se inventan nada.

Vale la pena que lo pruebes: cambia la línea de la pregunta para que use solo los patrones y
vuelve a correr la celda. Verás las preguntas legibles y los datos igual de protegidos.

Con eso resuelto, un registro así permite contestar lo que de verdad se pregunta cuando algo sale
mal: cuántas consultas tocaron documentos confidenciales, qué roles las hicieron, si alguien
preguntó lo mismo muchas veces con formulaciones distintas.

Fíjate también en que el identificador del usuario está convertido en huella y no guardado tal
cual. Eso permite saber que dos consultas vienen de la misma persona sin conservar su correo en
el registro. Es el principio de guardar solo lo mínimo necesario, que es lo que pide cualquier
normativa de protección de datos y además lo que conviene: lo que no guardas no se te puede
filtrar.

In [19]:
print("Revisión de la bitácora, del tipo que haría una auditoría:\n")

# Las tres revisiones que haría una auditoría: quién tocó lo confidencial, cuánto
# consultó cada rol, y si los registros están intactos.
confidenciales = [e for e in BITACORA if "confidencial" in e["niveles"]]
print(f"  consultas que tocaron documentos confidenciales: {len(confidenciales)}")
for e in confidenciales:
    print(f"     rol {e['rol']} -> {e['documentos']}")

por_rol = {}
for e in BITACORA:
    por_rol[e["rol"]] = por_rol.get(e["rol"], 0) + 1
print(f"\n  consultas por rol: {por_rol}")

print("\n  comprobación de integridad de cada entrada:")
for e in BITACORA:
    # Se recalcula la huella sin incluirla a ella misma y se compara con la guardada.
    copia = {k: v for k, v in e.items() if k != "huella"}
    recalculada = hashlib.sha256(
        json.dumps(copia, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()[:16]
    estado = "intacta" if recalculada == e["huella"] else "ALTERADA"
    print(f"     {e['momento']}  {estado}")

Revisión de la bitácora, del tipo que haría una auditoría:

  consultas que tocaron documentos confidenciales: 1
     rol gerente -> ['CON-MAR', 'PRO-DES']

  consultas por rol: {'cliente': 2, 'gerente': 1}

  comprobación de integridad de cada entrada:
     2026-08-07T18:53:19+00:00  intacta
     2026-08-07T18:53:20+00:00  intacta
     2026-08-07T18:53:20+00:00  intacta


## 17. Lo que te llevas

**Sobre detectar datos personales.** Ninguna herramienta sola alcanza. Los modelos de entidades
ven nombres y no ven números; las expresiones regulares ven números y no ven nombres. La
combinación es lo que funciona. Y la herramienta especializada más usada de la industria no
reconoce el DNI ni el RUC peruanos, porque no fue hecha para este mercado: si operas en la
región, tienes que verificar qué detecta y agregar lo que falte.

**Sobre medirlo.** Un conjunto de textos con los datos marcados a mano es lo que convierte
"nuestro detector funciona bien" en una cifra defendible. Sin eso no puedes responder qué se te
escapa, y esa es justamente la pregunta que va a hacer quien te audite.

**Sobre cómo tachar.** Borrar el dato rompe la frase; reemplazarlo por un relleno la deja opaca;
sustituirlo por su tipo conserva el sentido. Las tres protegen igual, así que conviene la que
menos daña el texto.

**Sobre los fallos silenciosos.** El nombre de la clienta sobrevivió a la redacción y nada avisó.
El sistema tachó el saludo y dejó el nombre completo. Este tipo de fallo no aparece en ninguna
prueba automática que no compare contra datos marcados a mano.

**Sobre trasladar una solución.** El mismo redactor que funcionaba sobre los tickets destrozó las
preguntas del registro, porque una frase corta no le da contexto al modelo de entidades. Lo que
se calibra para un tipo de texto hay que volver a medirlo antes de usarlo en otro.

**Sobre el control de acceso.** El índice no sabe quién pregunta. Filtrar por metadatos antes de
buscar resuelve la fuga, pero deja un problema de servicio: el usuario recibe algo irrelevante en
lugar de una negativa clara. Hay que combinarlo con la honestidad de la práctica anterior, y
decidir a propósito qué se le dice a quien pregunta por algo que no le corresponde.

**Sobre el registro.** Sin bitácora no se puede demostrar nada, y la bitácora es a su vez un
riesgo: se guarda redactada, con el usuario convertido en huella y con un hash que permita
detectar alteraciones.

Y el hilo que atraviesa toda la práctica: en privacidad, lo que no se mide se supone, y suponer
sale caro. Todo lo que viste aquí se puede automatizar salvo una cosa, que es marcar a mano el
conjunto contra el cual medir. Esa tarde de trabajo es la que sostiene todo lo demás.